# Cambrian-P and Cambrian-S: ask questions about your own videos

Runs [Cambrian-P](https://github.com/cambrian-mllm/cambrian-p) on one video or a whole folder of videos in your Google Drive, with one model or all five released models. For each video it answers your questions and plots the camera path the model predicts, then compares the models side by side.

**Before you start:** Runtime → Change runtime type → **A100 GPU**. Each 7B model needs about 15 GB for its weights alone, so the free T4 is too small. An L4 (24 GB) may work with fewer frames (set `NUM_FRAMES` to 32).

**How to use**
1. Put your videos in one Drive folder and set `VIDEO_PATH` to that folder (or to a single video file). Fill in the questions in the settings cell and run it.
2. Run the rest of the cells in order. Skip cell 4 (the Drive backup) unless cell 3 just did a fresh install.
3. The first time takes about 15 minutes of setup. Each model then downloads once (about 16 GB, a few minutes) and loads once to process every video. All five models take very roughly 30–60 minutes plus a few minutes per video per model.

`RUN_ALL_MODELS` (on by default) runs all five models; untick it to run only `MODEL`. Downloaded models are kept, so reruns in the same session skip the download. Within one session you can change the videos or questions and rerun only the settings cell and the last three cells.

**What you get**, in `OUTPUT_DIR` on your Drive:
- `comparison/`: every model's answers side by side (`answers_side_by_side.csv`), all answers with a score where you gave a true answer (`all_answers.csv`), a score per model (`scores.csv`), and one plot per video with all models' camera paths overlaid (`<video>_trajectories.png`).
- One folder per model, containing `results.csv` (one row per question) and one folder per video with:
  - `frames.jpg`: some of the frames the model saw. Check that they are upright.
  - `trajectory.png`: the predicted camera path, from above and in 3D.
  - `poses.npz` and `trajectory_tum.txt`: the raw per-frame camera poses.

**Cambrian-S baseline:** with `RUN_CAMBRIAN_S` ticked, the notebook also runs [Cambrian-S](https://github.com/cambrian-mllm/cambrian-s), the same architecture without camera tokens or a pose head, on the same videos and questions. Its answers appear as extra columns in the comparison and are scored the same way, so you can see what pose grounding changes.
- `Cambrian-S-7B` is the model the Cambrian-P paper compares against (its "no-pose counterpart"). `Cambrian-S-7B-S3` is the earlier checkpoint that Cambrian-P training starts from (`cambrianp/scripts/Cambrian-P-7B.sh`), so it shows the model before any spatial training. `Both` runs the two.
- It runs with the authors' own Cambrian-S code and evaluation settings (`cambrian-s/lmms-eval/evaluate_all_in_one.sh`): one frame per second, up to 64 frames. Cambrian-P instead spreads 32–128 frames over the clip, so on a 25-second clip Cambrian-S sees 25 frames. That is each model as published, not a frame-for-frame match.
- Cambrian-S needs different library versions (`transformers` 4.39 vs Cambrian-P's newer build), so cell 3b installs it in a separate environment (about 10 minutes, once; cell 4 backs it up too).
- Cambrian-S predicts no camera path, so it has no `trajectory.png` and doesn't appear in the trajectory overlays.
- To add Cambrian-S without rerunning the Cambrian-P models, untick `RUN_CAMBRIAN_P`. The comparison still picks up the Cambrian-P answers already saved in `OUTPUT_DIR`, as long as the questions haven't changed.

**Recording tips:** iPhone `.MOV` files (HEVC, portrait or landscape) load as they are, upright. Landscape is still better: every frame is padded to a square, so a portrait clip reaches the model at lower detail. Glance at `frames.jpg` once per video; if frames are ever sideways, the video was read by the fallback reader (PyAV), which ignores the phone's rotation, and converting the clip to an H.264 mp4 fixes it.

**Scores** follow VSI-Bench (lmms-eval/lmms_eval/tasks/vsibench/utils.py): a numeric true answer is scored by mean relative accuracy (1.0 = within 5%, partial credit down to within 50%), a letter or one-word answer by exact match on the model's first word, and longer text by exact match. Questions without a true answer are not scored.

**Camera poses:** every checkpoint predicts them. The paper's pose results use `Cambrian-P-7B-Mix-MA` (doc/evaluation.md), so try it when the path matters most. The path starts at the first frame. Distances are in the model's own relative units, not meters, so compare shapes (does a loop close, is a turn 90°) rather than lengths.

**Check the load worked:** the run log should contain a `token_num:` line, `[cam-load] ✓ Loaded model.rec_head.camera_tokens`, and `SUCCESS: All weights in new_state_dict match the loaded model.` Without them, the camera tokens are not being used.

In [ ]:
# @title 1. Settings
VIDEO_PATH = "/content/drive/MyDrive/Cambrian-P/Video_files"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/Cambrian-P/results"  # @param {type:"string"}
RUN_ALL_MODELS = True  # @param {type:"boolean"}
MODEL = "Cambrian-P-7B"  # @param ["Cambrian-P-7B", "Cambrian-P-7B-32f", "Cambrian-P-7B-Mix-MA", "Cambrian-P-7B-Mix-3R", "Cambrian-P-7B-Mix-CamS"]
RUN_CAMBRIAN_P = True  # @param {type:"boolean"}
RUN_CAMBRIAN_S = True  # @param {type:"boolean"}
CAMBRIAN_S_MODEL = "Cambrian-S-7B"  # @param ["Cambrian-S-7B", "Cambrian-S-7B-S3", "Both"]
NUM_FRAMES = 0  # @param {type:"integer"}
DELETE_MODELS_AFTER_RUN = False  # @param {type:"boolean"}
BACKUP_PATH = "/content/drive/MyDrive/cambrianp_env.tar"  # @param {type:"string"}
BACKUP_INCLUDES_MODEL = False  # @param {type:"boolean"}
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"  # @param {type:"string"}
CAMBRIAN_S_REPO_URL = "https://github.com/cambrian-mllm/cambrian-s.git"  # @param {type:"string"}

# VIDEO_PATH is a single video or a folder; every video in the folder is processed.
# RUN_ALL_MODELS runs all five models in turn; untick it to run only MODEL.
# RUN_CAMBRIAN_P off skips running Cambrian-P but still compares its answers already saved in OUTPUT_DIR.
# RUN_CAMBRIAN_S adds Cambrian-S (no camera tokens) as a baseline; see "Cambrian-S baseline" above.
# DELETE_MODELS_AFTER_RUN frees about 16 GB of disk per model, but reruns download them again.

# These two helpers phrase a question the way VSI-Bench does (lmms-eval/lmms_eval/tasks/vsibench/utils.py),
# which is the format the model was trained and scored on.
def number_q(question):
    """For a number or a single word: counts, distances (m), object sizes (cm), room area (m²)."""
    return f"These are frames of a video.\n{question}\nPlease answer the question using a single word or phrase."

def choice_q(question, options):
    """Multiple choice: relative direction or distance, route planning, appearance order."""
    lettered = "\n".join(f"{letter}. {option}" for letter, option in zip("ABCDEFGH", options))
    return f"These are frames of a video.\n{question}\nOptions:\n{lettered}\nAnswer with the option's letter from the given choices directly."

# Asked on every video. Each entry is a question, or [question, true answer] to compare against.
QUESTIONS = [
    "Describe the path the camera takes through the scene.",
    number_q("How many chairs are in this room?"),
]

# Extra questions for particular videos, keyed by file name. Leave it as {} if you have none.
PER_VIDEO_QUESTIONS = {
    "IMG_1068.MOV": [
    [choice_q("What number is clearly printed in white on the grey trash bin shown near the beginning of the video?",
              ["1026", "1062", "1082", "1602"]), "B"],
    [choice_q("What is the make of the black SUV parked directly next to the pile of black garbage bags on the sidewalk?",
              ["Audi", "BMW", "Lexus", "Honda"]), "C"],
    ],
    "IMG_1068.mov": [
    [choice_q("Imagine you are a robot standing on the sidewalk facing the black trash bags on the curb. How would you navigate to enter the glass doors seen at the end of the video?",
              ["Turn 90 degrees right, walk forward down the sidewalk, turn 90 degrees right, and walk up the stairs.",
                "Turn 90 degrees left, walk forward down the sidewalk, turn 90 degrees right, and walk up the stairs.",
                "Turn 180 degrees, walk across the street, and turn left.",
                "Walk backward, turn 90 degrees left, and walk up the stairs."]), "A"],
    [choice_q("Imagine you are a robot standing on the sidewalk directly facing the recycling and cardboard pile in the alcove. How would you navigate to the white van parked on the street?",
              ["Turn 90 degrees left, walk forward down the sidewalk, and the van will be parked on your left.",
                "Turn 90 degrees right, walk forward down the sidewalk, and the van will be parked on your right.",
                "Walk straight forward down into the recycling area.",
                "Turn 180 degrees, walk to the curb, and the van will be directly in front of you."]), "A"],
    ],
    "IMG_1067.MOV": [
    [choice_q("Imagine you are a robot standing on the sidewalk directly facing the EXIT-8 doors. How would you navigate back to the brown trash cans seen earlier in the video?",
              ["Turn 90 degrees right and walk straight down the sidewalk.",
                "Turn 90 degrees left and walk straight down the sidewalk.",
                "Turn 180 degrees, cross the street, and walk down the opposite sidewalk.",
                "Continue walking forward in the camera's original direction."]), "B"],
    [choice_q("According to the parking signs on the side of the street where the camera is walking, what are the hours for street cleaning?",
              ["8:30am-10:00am",
                "9:30am-11:00am",
                "11:30am-1pm",
                "1:00pm-2:30pm"]), "C"],
    ],
    "IMG_1066.MOV": [
    [choice_q("What brand name is printed on the white box of cat litter sitting next to the stainless steel trash can?",
              ["Fresh Step", "Arm & Hammer", "Tidy Cats", "Dr. Elsey's"]), "C"],
    [choice_q("What color are the heat-resistant gloves hanging on the wall above the stove?",
              ["Blue", "Red", "Green", "Yellow"]), "B"],
    [choice_q("What brand name is printed on the lid of the round container on the black dining table, next to the paper towel roll?",
              ["Great Value", "Trader Joe's", "Member's Mark", "Kirkland"]), "D"],
    [choice_q("What is inside the clear plastic bag lying on the cardboard box at the end of the video?",
              ["A roll of packing tape", "A spool of 3D printer filament", "A frying pan", "A Bluetooth speaker"]), "B"],
    [choice_q("When the cat appears near the washing machine, what is it doing?",
              ["Eating from bowls on the floor", "Sleeping on the counter", "Sitting in a laundry hamper", "Playing with a toy"]), "A"],
    [choice_q("What is covering the front of the refrigerator?",
              ["Takeout menus", "Children's drawings", "Nothing", "Instant photos"]), "D"],
    [choice_q("What word is printed in white on the black bag on the floor next to the stainless steel trash can?",
              ["Nike", "Gap", "Theory", "Puma"]), "C"],
    [choice_q("What brand are the white slide sandals with black stripes on the floor near the start of the video?",
              ["adidas", "Nike", "Puma", "Crocs"]), "A"],
    [choice_q("How many laundry hampers are in the video?",
              ["1", "2", "3", "4"]), "B"],
    [choice_q("What will be the first-time appearance order of the following categories in the video: "
              "laundry hamper, cat, trash can, paper towel roll?",
              ["cat, laundry hamper, paper towel roll, trash can",
               "trash can, laundry hamper, cat, paper towel roll",
               "laundry hamper, trash can, cat, paper towel roll",
               "laundry hamper, cat, trash can, paper towel roll"]), "D"],
    ],
    "IMG_1063.MOV": [
        [choice_q("At the end of the video, the camera stops at two waste bins. What word is printed in white on the black band of the bin on the left?",
                  ["WASTE", "LANDFILL", "TRASH", "GARBAGE"]), "C"],
        [choice_q("What color is the band labeled RECYCLE on the bin on the right?",
                  ["Green", "Blue", "Yellow", "Red"]), "B"],
        [choice_q("What color is the liner bag inside the TRASH bin?",
                  ["Black", "Clear", "Blue", "Green"]), "A"],
        [choice_q("While walking down the corridor, on which side of the camera are the tall slatted decorative panels?",
                  ["Left", "Both sides", "Overhead", "Right"]), "D"],
        [choice_q("What color are the slatted decorative panels lining the corridor?",
                  ["Dark brown", "White", "Pale yellow", "Black"]), "C"],
        [choice_q("One of the women walking toward the camera wears a cropped hooded sweatshirt. What color is it?",
                  ["Cream", "Red", "Black", "Green"]), "A"],
        [choice_q("How does the person filming get through the glass door?",
                  ["Swipes a card", "Pushes it open by hand", "It slides open automatically", "Someone holds it open"]), "B"],
        [choice_q("What color is the frame of the glass door the camera passes through?",
                  ["Silver", "White", "Wood brown", "Black"]), "D"],
        [choice_q("What are most of the people inside the glass-walled room doing?",
                  ["Standing in line", "Sitting at long tables", "Exercising", "Watching a presentation"]), "B"],
    ],
}

# NUM_FRAMES = 0 uses the frame count each model was evaluated with (doc/evaluation.md).
DEFAULT_FRAMES = {
    "Cambrian-P-7B": 128,
    "Cambrian-P-7B-32f": 32,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-Mix-3R": 32,
    "Cambrian-P-7B-Mix-CamS": 128,
}
# Cambrian-S takes one frame per second, up to this many (cambrian-s/lmms-eval/evaluate_all_in_one.sh).
CAMBRIAN_S_FRAMES = {"Cambrian-S-7B": 64, "Cambrian-S-7B-S3": 64}
CAMBRIAN_S_MODELS = (list(CAMBRIAN_S_FRAMES) if CAMBRIAN_S_MODEL == "Both" else [CAMBRIAN_S_MODEL]) if RUN_CAMBRIAN_S else []
# Every model to compare; cell 5 runs the Cambrian-P ones only if RUN_CAMBRIAN_P is ticked.
MODELS = (list(DEFAULT_FRAMES) if RUN_ALL_MODELS else [MODEL]) + CAMBRIAN_S_MODELS

def frames_for(name):
    return NUM_FRAMES or DEFAULT_FRAMES.get(name) or CAMBRIAN_S_FRAMES[name]

import json, os
os.environ.update({
    "BACKUP_PATH": BACKUP_PATH, "BACKUP_INCLUDES_MODEL": str(int(BACKUP_INCLUDES_MODEL)),
    "REPO_URL": REPO_URL, "CAMBRIAN_S_REPO_URL": CAMBRIAN_S_REPO_URL,
    "RUN_CAMBRIAN_S": str(int(bool(CAMBRIAN_S_MODELS))),
    # Keep uv's Python inside /content so the Drive backup contains everything the venv needs.
    "UV_PYTHON_INSTALL_DIR": "/content/python",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
})
with open("/content/questions.json", "w") as f:
    json.dump({"all": QUESTIONS, "per_video": PER_VIDEO_QUESTIONS}, f)
for name in MODELS:
    if name in CAMBRIAN_S_MODELS:
        print(f"{name}: 1 frame per second, up to {frames_for(name)} frames")
    else:
        print(f"{name}: {frames_for(name)} frames" + ("" if RUN_CAMBRIAN_P else " (not rerun; earlier answers are compared)"))
print(f"{len(QUESTIONS)} question(s) for every video, extra questions for {len(PER_VIDEO_QUESTIONS)} video(s)")

In [ ]:
# @title 2. Check the GPU and mount Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Runtime → Change runtime type → A100 GPU, then run again.")
print("GPU:", gpu)
if int(gpu.split(",")[-1].split()[0]) < 30000:
    print("Warning: under 30 GB of GPU memory. If you run out of memory, set NUM_FRAMES to 32 in cell 1.")

from google.colab import drive
drive.mount("/content/drive")
VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
if os.path.isdir(VIDEO_PATH):
    found = sorted(f for f in os.listdir(VIDEO_PATH) if f.lower().endswith(VIDEO_EXTS))
    print(f"{len(found)} video(s) in {VIDEO_PATH}:", ", ".join(found) or "none")
    for name in sorted(set(PER_VIDEO_QUESTIONS) - set(found)):
        print(f"Warning: PER_VIDEO_QUESTIONS names {name!r}, which is not in the folder (names must match exactly).")
elif os.path.isfile(VIDEO_PATH):
    print("Video:", VIDEO_PATH)
else:
    print(f"Warning: nothing found at {VIDEO_PATH}. Check the path in cell 1 (browse it in the Files panel on the left).")

In [ ]:
%%bash
# @title 3. Set up the environment (skips itself if already done)
set -e
cd /content
if /content/venv/bin/python -c "import torch" 2>/dev/null && [ -d /content/cambrian-p ]; then
    echo "Environment already set up in this session."
elif [ -f "$BACKUP_PATH" ]; then
    echo "Restoring environment from $BACKUP_PATH ..."
    tar xf "$BACKUP_PATH" -C /content
    echo "Restored."
else
    echo "Fresh install (about 10-15 minutes) ..."
    [ -d cambrian-p ] || git clone -q "$REPO_URL" cambrian-p
    pip install -q uv
    # The repo is pinned to Python 3.11, which may differ from Colab's own Python.
    uv venv --seed --python 3.11 /content/venv
    source /content/venv/bin/activate
    cd cambrian-p
    # Same steps and order as the README's "Environment Preparation".
    pip install -q -e ./vggt
    pip install -q hydra-core tensorboard iopath wcmatch fvcore
    pip install -q --upgrade pip
    pip install -q -e ".[train]"
    pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
        --index-url https://download.pytorch.org/whl/cu121
    pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
    # Used by lmms-eval's model wrapper. flash-attn is skipped: the wrapper always uses PyTorch's
    # built-in attention (sdpa) on torch >= 2.1.2, so it would never be used.
    pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
    echo "Install finished."
fi
# pyarrow: newer releases need NumPy 2, but the repo pins NumPy 1.26.1. Also fixes backups made before this pin.
/content/venv/bin/pip install -q pyarrow==15.0.2


In [ ]:
%%bash
# @title 3b. Set up Cambrian-S (only with RUN_CAMBRIAN_S; skips itself if already done)
set -e
cd /content
if [ "$RUN_CAMBRIAN_S" != "1" ]; then
    echo "RUN_CAMBRIAN_S is off; nothing to do."
elif /content/venv_s/bin/python -c "import cambrian" 2>/dev/null; then
    echo "Cambrian-S environment already set up."
else
    echo "Installing Cambrian-S in its own environment (about 10 minutes) ..."
    [ -d cambrian-s ] || git clone -q "$CAMBRIAN_S_REPO_URL" cambrian-s
    pip install -q uv
    # Python 3.10 and the install steps from the Cambrian-S evaluation README (cambrian-s/lmms-eval/README.md).
    uv venv --seed --python 3.10 /content/venv_s
    /content/venv_s/bin/pip install -q -e ./cambrian-s
    # Installing lmms-eval next would move these packages to the versions below, which is the environment
    # the authors evaluate with. Only these are installed here: lmms-eval itself isn't needed and pulls in
    # TensorFlow. pip may warn that cambrian asked for other versions; that is expected.
    /content/venv_s/bin/pip install -q torch==2.5.1 torchvision==0.20.1 transformers==4.39.2 accelerate==0.34.2 \
        diffusers==0.27.2 open_clip_torch==2.32.0 huggingface-hub==0.25.2 numpy==1.26.4 hf_transfer
    /content/venv_s/bin/python -c "import cambrian, torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"
    echo "Install finished."
fi

In [ ]:
%%bash
# @title 4. (Optional, run once after a fresh install) Save the environment to Drive for faster future sessions
# Without models the archive is several GB; with them, add about 16 GB per downloaded model (needs paid Drive storage).
set -e
cd /content
EXCLUDE="--exclude=cambrian-p/ckpts"
[ "$BACKUP_INCLUDES_MODEL" = "1" ] && EXCLUDE=""
for d in python venv cambrian-p; do
    if [ ! -d "$d" ]; then
        echo "Nothing to save: /content/$d is missing. Run cell 3 first; your existing backup was left untouched."
        exit 1
    fi
done
# Write to a temporary file first so a failed or interrupted save can't damage an existing backup.
echo "Writing $BACKUP_PATH ..."
# The Cambrian-S environment (cell 3b) is included when it exists.
EXTRA=""
for d in venv_s cambrian-s; do [ -d "$d" ] && EXTRA="$EXTRA $d"; done
tar cf "$BACKUP_PATH.partial" $EXCLUDE python venv cambrian-p $EXTRA
mv "$BACKUP_PATH.partial" "$BACKUP_PATH"
ls -lh "$BACKUP_PATH"

In [ ]:
%%writefile /content/run_videos.py
"""Answer questions about one or more videos with Cambrian-P and plot each video's predicted camera path.

The model is loaded once. For each video it writes, under OUTPUT_DIR/<video name>/:
  answers.json        questions, true answers (if given) and the model's answers
  frames.jpg          a few of the frames the model saw (check the orientation here)
  trajectory.png      bird's-eye and 3D view of the predicted camera path
  poses.npz           per-frame camera-to-world matrices, focal lengths, timestamps
  trajectory_tum.txt  the same path in TUM format, for tools like `evo`
and OUTPUT_DIR/results.csv with one row per question across all videos.
"""
import argparse
import csv
import glob
import json
import os
import traceback

# Colab sets MPLBACKEND to its notebook-inline backend, which %%bash passes on to this script.
# This venv doesn't have that backend, so matplotlib fails on import; this script only saves files.
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from decord import VideoReader, cpu
from PIL import Image
from scipy.spatial.transform import Rotation

import cambrianp.utils as cambrianp_utils
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from cambrianp.conversation import conv_templates
from cambrianp.mm_utils import tokenizer_image_token
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")

ap = argparse.ArgumentParser()
ap.add_argument("--model_path", required=True)
ap.add_argument("--videos", required=True, help="a video file, or a folder of videos")
ap.add_argument("--questions_file", required=True)
ap.add_argument("--output_dir", required=True)
ap.add_argument("--num_frames", type=int, default=128)
args = ap.parse_args()

if os.path.isdir(args.videos):
    videos = sorted(p for p in glob.glob(os.path.join(args.videos, "*")) if p.lower().endswith(VIDEO_EXTS))
else:
    videos = [args.videos]
if not videos:
    raise SystemExit(f"No videos found at {args.videos}")

# {"all": [...], "per_video": {"name.mp4": [...]}}; each entry is a question or [question, true answer].
with open(args.questions_file) as f:
    question_spec = json.load(f)


def questions_for(video):
    entries = question_spec.get("all", []) + question_spec.get("per_video", {}).get(os.path.basename(video), [])
    return [(e, "") if isinstance(e, str) else (e[0], str(e[1])) for e in entries]


# For any checkpoint path containing "cambrian", the loader converts the config to LLaVA-OneVision
# style (cambrianp/utils.py). That conversion is meant for starting training from Cambrian-S. Released
# Cambrian-P checkpoints were saved already converted, so converting again overwrites vocab_size
# (152064 vs the weights' 151672) and deletes the camera-head settings, which silently drops the
# per-frame camera tokens. Keep the checkpoint's config as saved, and switch the camera head on
# (use_camera_tokens only takes effect with load_rec_model, llava_arch.py:556; the pose eval sets it too).
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        # Path to the authors' VGGT init file; the trained head weights come from the checkpoint itself.
        config.rec_model_path = "None"
    return config

cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config

# Flags from doc/evaluation.md.
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")
pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id


def ask(pixels, image_size, video, question, gen_kwargs=None):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    return m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", dict(gen_kwargs or {}), video)[0]


def decode_poses(preds):
    """Camera-to-world 4x4 matrices and focal lengths, decoded the same way as extract_pose_llava in
    cambrianp/eval/relpose/unified_pose_eval.py. Frame 0 is the origin; OpenCV axes (x right, y down,
    z forward). Translation is in the model's normalized units, not meters."""
    extrinsic, intrinsic = pose_encoding_to_extri_intri(preds["pose_enc"].float(), (384, 384))
    extrinsic, intrinsic = extrinsic[0].cpu().numpy(), intrinsic[0].cpu().numpy()
    c2w = np.tile(np.eye(4), (len(extrinsic), 1, 1))
    R, t = extrinsic[:, :3, :3], extrinsic[:, :3, 3]
    c2w[:, :3, :3] = R.transpose(0, 2, 1)
    c2w[:, :3, 3] = -np.einsum("nji,nj->ni", R, t)
    return c2w, intrinsic[:, 0, 0]


def plot_trajectory(c2w, times, title, path):
    pos = c2w[:, :3, 3]
    fwd = c2w[:, :3, 2]  # camera viewing direction in world coordinates
    fig = plt.figure(figsize=(13, 6))

    # Bird's-eye view: x (right) against z (forward) of the first frame, for a camera held level.
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(pos[:, 0], pos[:, 2], color="0.75", lw=1, zorder=1)
    sc = ax.scatter(pos[:, 0], pos[:, 2], c=times, cmap="viridis", s=14, zorder=2)
    step = max(1, len(pos) // 16)
    span = max(np.ptp(pos[:, 0]), np.ptp(pos[:, 2]), 1e-6)
    ax.quiver(pos[::step, 0], pos[::step, 2], fwd[::step, 0], fwd[::step, 2], color="tab:orange",
              angles="xy", scale_units="xy", scale=1 / (0.08 * span), width=0.004, zorder=3)
    ax.scatter(*pos[0, [0, 2]], marker="o", s=120, facecolors="none", edgecolors="green", lw=2, label="start", zorder=4)
    ax.scatter(*pos[-1, [0, 2]], marker="X", s=120, color="red", label="end", zorder=4)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("x: right of the first frame")
    ax.set_ylabel("z: forward from the first frame")
    ax.set_title("Bird's-eye view (arrows = where the camera points)")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
    fig.colorbar(sc, ax=ax, label="time in video (s)")

    # 3D view; y is negated so "up" points up.
    ax3 = fig.add_subplot(1, 2, 2, projection="3d")
    ax3.plot(pos[:, 0], pos[:, 2], -pos[:, 1], color="0.6", lw=1)
    ax3.scatter(pos[:, 0], pos[:, 2], -pos[:, 1], c=times, cmap="viridis", s=10)
    ax3.scatter(*pos[0, [0, 2]], -pos[0, 1], color="green", s=60)
    ax3.scatter(*pos[-1, [0, 2]], -pos[-1, 1], color="red", marker="X", s=60)
    ax3.set_xlabel("x (right)")
    ax3.set_ylabel("z (forward)")
    ax3.set_zlabel("up")
    # True proportions, so small up/down jitter is not stretched to fill the box.
    ax3.set_box_aspect(np.maximum(np.ptp(pos[:, [0, 2, 1]], axis=0), 0.05 * span))
    ax3.set_title("3D view")

    fig.suptitle(f"{title}  (units are relative, not meters)")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def save_frames_preview(frames, path, count=8):
    picks = np.linspace(0, len(frames) - 1, min(count, len(frames)), dtype=int)
    thumbs = [Image.fromarray(frames[i]) for i in picks]
    for t in thumbs:
        t.thumbnail((320, 320))
    w, h = max(t.width for t in thumbs), max(t.height for t in thumbs)
    cols = 4
    sheet = Image.new("RGB", (cols * w, -(-len(thumbs) // cols) * h), "white")
    for k, t in enumerate(thumbs):
        sheet.paste(t, ((k % cols) * w, (k // cols) * h))
    sheet.save(path, quality=85)


def run_video(video, out_dir):
    raw = m._load_video_frames(video, m.max_frames_num)
    # Timestamps of the frames load_video samples (the same uniform spacing over the whole clip).
    vr = VideoReader(video, ctx=cpu(0))
    frame_idx = np.linspace(0, len(vr) - 1, len(raw), dtype=int)
    times = frame_idx / vr.get_avg_fps()
    del vr
    print(f"{len(raw)} frames of {raw.shape[2]}x{raw.shape[1]} over {times[-1]:.1f}s", flush=True)
    save_frames_preview(raw, os.path.join(out_dir, "frames.jpg"))

    # Same frame preprocessing as CambrianP.generate_until for video inputs.
    image_size = list(raw.shape[1:3])
    bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
    frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in raw]
    pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()

    # The model computes camera poses once per generate() and then caches them on itself
    # (llava_qwen.py, "rec_head_preds"), so clear the previous video's poses first.
    m.model.rec_head_preds = None
    rows = []
    questions = questions_for(video)
    for question, truth in questions:
        answer = ask(pixels, image_size, video, question)
        print(f"\nQ: {question}\nA: {answer}" + (f"\n   (true answer: {truth})" if truth else ""), flush=True)
        rows.append({"video": os.path.basename(video), "question": question, "true_answer": truth, "model_answer": answer})
    if not questions:
        # No questions for this video: run one short pass just to get the poses.
        ask(pixels, image_size, video, "Describe this scene.", {"max_new_tokens": 1})
        rows.append({"video": os.path.basename(video), "question": "", "true_answer": "", "model_answer": "(no questions; poses only)"})
    with open(os.path.join(out_dir, "answers.json"), "w") as f:
        json.dump(rows, f, indent=2)

    # The camera tokens sit with the frames, before the question, so the poses do not depend on
    # which question was asked.
    preds = m.model.rec_head_preds
    if preds is None or "pose_enc" not in preds:
        print("Warning: the model returned no camera poses for this video (see the log above).")
        return rows
    c2w, focal = decode_poses(preds)
    np.savez(os.path.join(out_dir, "poses.npz"), c2w=c2w, focal=focal, frame_idx=frame_idx, times=times)
    quat = Rotation.from_matrix(c2w[:, :3, :3]).as_quat()  # x, y, z, w
    np.savetxt(os.path.join(out_dir, "trajectory_tum.txt"), np.column_stack([times, c2w[:, :3, 3], quat]), fmt="%.6f",
               header="timestamp tx ty tz qx qy qz qw")
    plot_trajectory(c2w, times, os.path.basename(video), os.path.join(out_dir, "trajectory.png"))
    print(f"Saved trajectory plot to {out_dir}/trajectory.png")
    return rows


os.makedirs(args.output_dir, exist_ok=True)
all_rows = []
for n, video in enumerate(videos, 1):
    name = os.path.splitext(os.path.basename(video))[0]
    print(f"\n===== [{n}/{len(videos)}] {os.path.basename(video)} =====", flush=True)
    out_dir = os.path.join(args.output_dir, name)
    os.makedirs(out_dir, exist_ok=True)
    try:
        all_rows += run_video(video, out_dir)
    except Exception:
        # Keep going with the other videos; the error is also recorded in results.csv.
        traceback.print_exc()
        all_rows.append({"video": os.path.basename(video), "question": "", "true_answer": "",
                         "model_answer": "ERROR: " + traceback.format_exc().strip().splitlines()[-1]})
    torch.cuda.empty_cache()

with open(os.path.join(args.output_dir, "results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["video", "question", "true_answer", "model_answer"])
    writer.writeheader()
    writer.writerows(all_rows)
print(f"\nDone: {len(videos)} video(s). Answers for all of them are in {args.output_dir}/results.csv")

In [ ]:
%%writefile /content/run_videos_cambrian_s.py
"""Answer questions about one or more videos with Cambrian-S, using the authors' own inference code.

Frame sampling, preprocessing, prompt and generation follow cambrian-s/lmms-eval/demo.py and
lmms_eval/models/simple/cambrians.py, with the settings from cambrian-s/lmms-eval/evaluate_all_in_one.sh,
so the answers match how Cambrian-S is evaluated. Output layout matches run_videos.py (answers.json and
frames.jpg per video, results.csv), so the comparison cell can read both. Cambrian-S predicts no camera poses.
"""
import argparse
import csv
import glob
import json
import os
import traceback

import numpy as np
import torch
from decord import VideoReader, cpu
from PIL import Image

from cambrian.constants import DEFAULT_IM_END_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from cambrian.conversation import conv_templates
from cambrian.mm_utils import expand2square, get_model_name_from_path, tokenizer_image_token
from cambrian.model.builder import load_pretrained_model

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")

ap = argparse.ArgumentParser()
ap.add_argument("--model_path", required=True)
ap.add_argument("--videos", required=True, help="a video file, or a folder of videos")
ap.add_argument("--questions_file", required=True)
ap.add_argument("--output_dir", required=True)
ap.add_argument("--max_frames", type=int, default=64)
ap.add_argument("--max_new_tokens", type=int, default=1024)
args = ap.parse_args()

if os.path.isdir(args.videos):
    videos = sorted(p for p in glob.glob(os.path.join(args.videos, "*")) if p.lower().endswith(VIDEO_EXTS))
else:
    videos = [args.videos]
if not videos:
    raise SystemExit(f"No videos found at {args.videos}")

# {"all": [...], "per_video": {"name.mp4": [...]}}; each entry is a question or [question, true answer].
with open(args.questions_file) as f:
    question_spec = json.load(f)


def questions_for(video):
    entries = question_spec.get("all", []) + question_spec.get("per_video", {}).get(os.path.basename(video), [])
    return [(e, "") if isinstance(e, str) else (e[0], str(e[1])) for e in entries]


tokenizer, model, image_processors, _ = load_pretrained_model(
    args.model_path, None, get_model_name_from_path(args.model_path), device_map="cuda:0")
# Settings from cambrian-s/lmms-eval/evaluate_all_in_one.sh (and the CambrianS wrapper's defaults).
model.config.video_max_frames = args.max_frames
model.config.video_fps = 1
model.config.video_force_sample = False
model.config.add_time_instruction = False
model.config.miv_token_len = 64
model.config.si_token_len = 729
model.config.image_aspect_ratio = "anyres"
model.config.anyres_max_subimages = 9
model.eval()
print(f"video_max_frames={args.max_frames}, video_fps=1, miv_token_len=64, si_token_len=729", flush=True)


def load_frames(video):
    """Frame sampling from process_video_with_decord: one frame per second, or max_frames spread
    evenly over the clip if that is fewer."""
    vr = VideoReader(video, ctx=cpu(0))
    total = len(vr)
    frame_idx = list(range(0, total, round(vr.get_avg_fps() / model.config.video_fps)))
    if model.config.video_max_frames > 0 and (len(frame_idx) > model.config.video_max_frames or model.config.video_force_sample):
        frame_idx = np.linspace(0, total - 1, model.config.video_max_frames, dtype=int).tolist()
    frames = vr.get_batch(frame_idx).asnumpy()
    times = np.array(frame_idx) / vr.get_avg_fps()
    vr.seek(0)
    return frames, times


def preprocess(frames):
    """process_videos for a single video: pad each frame to a square, then each vision tower's processor."""
    pil = [Image.fromarray(f, mode="RGB") for f in frames]
    tensors = []
    for processor in image_processors:
        bg = tuple(int(x * 255) for x in processor.image_mean)
        pixels = processor.preprocess([expand2square(im, bg) for im in pil], return_tensors="pt")["pixel_values"]
        tensors.append(torch.stack([pixels]).half().cuda())
    sizes = [(frames.shape[2], frames.shape[1], frames.shape[0])]  # W, H, T
    return tensors, sizes


def ask(tensors, sizes, question):
    if model.config.mm_use_im_start_end:
        question = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + "\n" + question
    else:
        question = DEFAULT_IMAGE_TOKEN * len(tensors) + "\n" + question
    conv = conv_templates["qwen_2"].copy()
    conv.append_message(conv.roles[0], question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).cuda()
    # Greedy decoding as in the wrapper. Its default of 16 new tokens is raised so open-ended answers
    # aren't cut off; letter and number answers are the same either way.
    with torch.inference_mode():
        out = model.generate(inputs=ids, images=tensors, image_sizes=sizes, use_cache=True, do_sample=False,
                             temperature=0, top_p=None, num_beams=1, max_new_tokens=args.max_new_tokens)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0].strip()


def save_frames_preview(frames, path, count=8):
    picks = np.linspace(0, len(frames) - 1, min(count, len(frames)), dtype=int)
    thumbs = [Image.fromarray(frames[i]) for i in picks]
    for t in thumbs:
        t.thumbnail((320, 320))
    w, h = max(t.width for t in thumbs), max(t.height for t in thumbs)
    cols = 4
    sheet = Image.new("RGB", (cols * w, -(-len(thumbs) // cols) * h), "white")
    for k, t in enumerate(thumbs):
        sheet.paste(t, ((k % cols) * w, (k // cols) * h))
    sheet.save(path, quality=85)


def run_video(video, out_dir):
    frames, times = load_frames(video)
    print(f"{len(frames)} frames of {frames.shape[2]}x{frames.shape[1]} over {times[-1]:.1f}s", flush=True)
    save_frames_preview(frames, os.path.join(out_dir, "frames.jpg"))
    tensors, sizes = preprocess(frames)
    rows = []
    for question, truth in questions_for(video):
        answer = ask(tensors, sizes, question)
        print(f"\nQ: {question}\nA: {answer}" + (f"\n   (true answer: {truth})" if truth else ""), flush=True)
        rows.append({"video": os.path.basename(video), "question": question, "true_answer": truth, "model_answer": answer})
    if not rows:
        rows.append({"video": os.path.basename(video), "question": "", "true_answer": "", "model_answer": "(no questions)"})
    with open(os.path.join(out_dir, "answers.json"), "w") as f:
        json.dump(rows, f, indent=2)
    return rows


os.makedirs(args.output_dir, exist_ok=True)
all_rows = []
for n, video in enumerate(videos, 1):
    name = os.path.splitext(os.path.basename(video))[0]
    print(f"\n===== [{n}/{len(videos)}] {os.path.basename(video)} =====", flush=True)
    out_dir = os.path.join(args.output_dir, name)
    os.makedirs(out_dir, exist_ok=True)
    try:
        all_rows += run_video(video, out_dir)
    except Exception:
        # Keep going with the other videos; the error is also recorded in results.csv.
        traceback.print_exc()
        all_rows.append({"video": os.path.basename(video), "question": "", "true_answer": "",
                         "model_answer": "ERROR: " + traceback.format_exc().strip().splitlines()[-1]})
    torch.cuda.empty_cache()

with open(os.path.join(args.output_dir, "results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["video", "question", "true_answer", "model_answer"])
    writer.writeheader()
    writer.writerows(all_rows)
print(f"\nDone: {len(videos)} video(s). Answers for all of them are in {args.output_dir}/results.csv")

In [ ]:
# @title 5. Run the model(s) on your videos
import shutil, subprocess, time

VENV_BIN = "/content/venv/bin"
REPO = "/content/cambrian-p"
env = dict(os.environ, PYTHONPATH=f"{REPO}/lmms-eval:{REPO}", PATH=f"{VENV_BIN}:{os.environ['PATH']}")
# Cambrian-S runs from its own environment (cell 3b), without Cambrian-P's code on the path.
env_s = dict(os.environ, MPLBACKEND="Agg", PATH=f"/content/venv_s/bin:{os.environ['PATH']}")
env_s.pop("PYTHONPATH", None)

def run(cmd, env=env):
    """Run a command, streaming its output; return (exit code, last lines of output)."""
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-40:]
    return p.wait(), "".join(tail)

status = {}
to_run = [m for m in MODELS if RUN_CAMBRIAN_P or m in CAMBRIAN_S_MODELS]
if CAMBRIAN_S_MODELS and not os.path.exists("/content/venv_s/bin/python"):
    raise SystemExit("Cambrian-S isn't set up yet: run cell 3b first.")
for name in to_run:
    frames = frames_for(name)
    print(f"\n{'#' * 25} {name} ({frames} frames) {'#' * 25}")
    t = time.time()
    ckpt = f"{REPO}/ckpts/{name}"
    if not os.path.isfile(f"{ckpt}/config.json"):
        free_gb = shutil.disk_usage("/content").free / 1e9
        if free_gb < 20:
            status[name] = f"skipped: only {free_gb:.0f} GB of disk free (needs about 16 GB)"
            print(f"{status[name]}. Tick DELETE_MODELS_AFTER_RUN in cell 1, then rerun cell 1 and this cell.")
            continue
        print(f"Downloading nyu-visionx/{name} (about 16 GB) ...")
        code_, _ = run([f"{VENV_BIN}/huggingface-cli", "download", f"nyu-visionx/{name}", "--local-dir", ckpt])
        if code_ != 0:
            status[name] = "download failed (see the log above)"
            continue
    # Each model runs in its own process, so its GPU memory is freed before the next one loads.
    common = ["--videos", VIDEO_PATH, "--questions_file", "/content/questions.json", "--output_dir", os.path.join(OUTPUT_DIR, name)]
    if name in CAMBRIAN_S_MODELS:
        code_, _ = run(["/content/venv_s/bin/python", "/content/run_videos_cambrian_s.py", "--model_path", ckpt,
                        "--max_frames", str(frames)] + common, env=env_s)
    else:
        code_, _ = run([f"{VENV_BIN}/python", "/content/run_videos.py", "--model_path", f"ckpts/{name}",
                        "--num_frames", str(frames)] + common)
    status[name] = "ok" if code_ == 0 else f"failed with exit code {code_} (see the log above)"
    if DELETE_MODELS_AFTER_RUN:
        shutil.rmtree(ckpt, ignore_errors=True)
    print(f"[{name}] {status[name]}, {(time.time() - t) / 60:.1f} min")

print("\nSummary:")
for name, s in status.items():
    print(f"  {name}: {s}")

In [ ]:
# @title 6. Compare the models
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_colwidth", None)
COMPARE_DIR = os.path.join(OUTPUT_DIR, "comparison")
os.makedirs(COMPARE_DIR, exist_ok=True)

if os.path.isdir(VIDEO_PATH):
    videos = sorted(f for f in os.listdir(VIDEO_PATH) if f.lower().endswith(VIDEO_EXTS))
else:
    videos = [os.path.basename(VIDEO_PATH)]

# Scoring as in VSI-Bench (lmms-eval/lmms_eval/tasks/vsibench/utils.py).
def first_word(answer):
    return answer.split(" ")[0].rstrip(".").strip()

def as_float(text):
    try:
        return float(text)
    except ValueError:
        return None

def score(answer, truth):
    """1.0/0.0 for exact matches, 0–1 mean relative accuracy for numbers, NaN if there is no true answer."""
    truth = str(truth).strip()
    if not truth:
        return np.nan
    if as_float(truth) is not None:
        target, pred = as_float(truth), as_float(first_word(answer))
        if pred is None:
            return 0.0
        if target == 0:
            return float(pred == 0)
        thresholds = np.linspace(0.5, 0.95, 10)
        return float(np.mean(abs(pred - target) / abs(target) <= 1 - thresholds))
    if len(truth.split()) == 1:
        return float(first_word(answer).lower() == truth.lower())
    return float(answer.strip().rstrip(".").lower() == truth.rstrip(".").lower())

def short(question):
    """The question itself, without the VSI-Bench framing added by number_q / choice_q."""
    lines = [l for l in question.splitlines() if l and l != "These are frames of a video."
             and not l.startswith(("Please answer the question", "Answer with the option"))]
    return " / ".join(lines)

# Collect every model's answers from the per-video answers.json files.
rows = []
for name in MODELS:
    for video in videos:
        path = os.path.join(OUTPUT_DIR, name, os.path.splitext(video)[0], "answers.json")
        if os.path.exists(path):
            for r in json.load(open(path)):
                rows.append(dict(r, model=name))
if not rows:
    raise SystemExit("No answers found yet. Run cell 5 first.")
answers = pd.DataFrame(rows)
answers["score"] = [score(a, t) for a, t in zip(answers["model_answer"], answers["true_answer"])]
answers["question_short"] = answers["question"].map(short)
answers[["video", "question", "true_answer", "model", "model_answer", "score"]].to_csv(
    os.path.join(COMPARE_DIR, "all_answers.csv"), index=False)

side = answers.pivot_table(index=["video", "question_short", "true_answer"], columns="model",
                           values="model_answer", aggfunc="first", sort=False)
side = side[[m for m in MODELS if m in side.columns]]
side.to_csv(os.path.join(COMPARE_DIR, "answers_side_by_side.csv"))

scored = answers.dropna(subset=["score"])
if len(scored):
    scores = scored.groupby("model")["score"].agg(score="mean", questions="count").reindex(
        [m for m in MODELS if m in set(scored["model"])])
    scores["score"] = (100 * scores["score"]).round(1)
    scores.to_csv(os.path.join(COMPARE_DIR, "scores.csv"))

def overlay_trajectories(video):
    """All models' predicted camera paths for one video, bird's-eye and 3D (axes as in trajectory.png)."""
    stem = os.path.splitext(video)[0]
    paths = {}
    for name in MODELS:
        p = os.path.join(OUTPUT_DIR, name, stem, "poses.npz")
        if os.path.exists(p):
            paths[name] = np.load(p)["c2w"][:, :3, 3]
    if not paths:
        return None
    fig = plt.figure(figsize=(14, 6))
    ax, ax3 = fig.add_subplot(1, 2, 1), fig.add_subplot(1, 2, 2, projection="3d")
    for (name, pos), color in zip(paths.items(), plt.cm.tab10.colors):
        ax.plot(pos[:, 0], pos[:, 2], color=color, lw=1.5, label=name)
        ax.scatter(*pos[0, [0, 2]], marker="o", s=60, facecolors="none", edgecolors=color, lw=2)
        ax.scatter(*pos[-1, [0, 2]], marker="X", s=60, color=color)
        ax3.plot(pos[:, 0], pos[:, 2], -pos[:, 1], color=color, lw=1.5)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("x: right of the first frame")
    ax.set_ylabel("z: forward from the first frame")
    ax.set_title("Bird's-eye view (○ start, ✕ end)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    ax3.set_xlabel("x (right)")
    ax3.set_ylabel("z (forward)")
    ax3.set_zlabel("up")
    ax3.set_title("3D view")
    # True proportions, so small up/down jitter is not stretched to fill the box.
    allpos = np.concatenate(list(paths.values()))
    span = max(np.ptp(allpos[:, 0]), np.ptp(allpos[:, 2]), 1e-6)
    ax3.set_box_aspect(np.maximum(np.ptp(allpos[:, [0, 2, 1]], axis=0), 0.05 * span))
    fig.suptitle(f"{video}: predicted camera paths (relative units; each model sets its own scale, so compare shapes)")
    fig.tight_layout()
    out = os.path.join(COMPARE_DIR, f"{stem}_trajectories.png")
    fig.savefig(out, dpi=120)
    plt.close(fig)
    return out

for video in videos:
    display(Markdown(f"## {video}"))
    stem = os.path.splitext(video)[0]
    preview = next((os.path.join(OUTPUT_DIR, m, stem, "frames.jpg") for m in MODELS
                    if os.path.exists(os.path.join(OUTPUT_DIR, m, stem, "frames.jpg"))), None)
    if preview:
        display(Image(preview, width=1000))
    overlay = overlay_trajectories(video)
    if overlay:
        display(Image(overlay, width=1100))
    if video in side.index.get_level_values("video"):
        display(side.loc[video])

if len(scored):
    display(Markdown("## Score per model (where you gave a true answer)"))
    display(scores)
missing = [m for m in MODELS if m not in set(answers["model"])]
if missing:
    print("No answers from:", ", ".join(missing))
print("Saved to", COMPARE_DIR)

In [ ]:
from google.colab import drive
drive.flush_and_unmount()